# Proyecto final PLN · Notebook 02: fine-tuning de BETO
**Caso Bodytech: priorización de reseñas negativas por sede** · Equipo: Jesús Joseph Suescun (Gerencia IA), Fabián David Sánchez (Machine Learning) y David Vallejo Villegas (Ciencia de Datos).

Este notebook ajusta **BETO** (`dccuchile/bert-base-spanish-wwm-cased`, un BERT preentrenado en español) con **exactamente la misma partición por sede y el mismo texto anonimizado** del notebook principal (`01_proyecto_final_bodytech.ipynb`). Así la comparación con TF-IDF y con la BiLSTM es justa.

**Cómo ejecutarlo (Google Colab):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4***.
2. *Entorno de ejecución → **Ejecutar todo***. Tarda unos 15 a 25 minutos con GPU.
3. Al final se descargan tres archivos: `beto_resumen.json`, `beto_probabilidades.csv` y `beto_contrafactuales.csv`. Súbalos al repositorio en la carpeta `resultados/beto/` (en GitHub: *Add file → Upload files*) y vuelva a ejecutar el notebook 01: su sección 15 los incorpora a la comparación.

**Reglas que se mantienen:** semilla 42, selección de la longitud máxima con **validación** y una sola evaluación en test, después de elegir.

Si Colab pide reiniciar el entorno después de instalar paquetes, reinícielo y vuelva a usar *Ejecutar todo*.

In [ ]:
# 0. Entorno: clonar el repositorio (datos y partición) y revisar la GPU
import sys, os, subprocess, importlib.util
if 'google.colab' in sys.modules and not os.path.exists('datos/procesado/corpus_bodytech_anonimizado.csv'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DavidVallejoV/proyecto-final-pln'], check=True)
    os.chdir('proyecto-final-pln')
    faltan = [p for p in ['transformers', 'datasets', 'accelerate'] if importlib.util.find_spec(p) is None]   # Colab trae casi todo
    if faltan:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan], check=True)
    print('Repositorio listo en', os.getcwd())

In [ ]:
import os, re, gc, json, time, shutil, inspect, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, transformers, datasets
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, set_seed)
from transformers.trainer_callback import PrinterCallback
from sklearn.metrics import f1_score, recall_score, log_loss, classification_report
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore'); os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
transformers.logging.set_verbosity_error(); transformers.logging.disable_progress_bar(); datasets.disable_progress_bars()

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos' / 'procesado').exists())
SALIDA = RAIZ / 'resultados' / 'beto'; SALIDA.mkdir(parents=True, exist_ok=True)
SEED = 42
CHECKPOINT = 'dccuchile/bert-base-spanish-wwm-cased'
LONGITUDES = [128, 256]                  # se comparan en validación; 256 cubre casi todas las reseñas
SEMILLAS_EXTRA = [1, 2]                  # la configuración elegida se repite con 2 semillas más para medir variabilidad
CLASES = ['negativa', 'neutra', 'positiva']; label2id = {c: i for i, c in enumerate(CLASES)}; id2label = {i: c for c, i in label2id.items()}
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DISPOSITIVO == 'cuda' else 'sin GPU'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| dispositivo:', DISPOSITIVO, '|', GPU)
if DISPOSITIVO == 'cpu':
    print('AVISO: sin GPU el entrenamiento puede tardar horas. Active la GPU en Colab antes de seguir.')

In [ ]:
# 1. Datos: el corpus anonimizado y la partición por sede del notebook principal
corpus = pd.read_csv(RAIZ / 'datos' / 'procesado' / 'corpus_bodytech_anonimizado.csv')
partes = {p: corpus[corpus.parte == p].reset_index(drop=True) for p in ['train', 'val', 'test', 'externo']}
print({p: len(t) for p, t in partes.items()})
y = {p: t.sentimiento.map(label2id).to_numpy() for p, t in partes.items()}

tok = AutoTokenizer.from_pretrained(CHECKPOINT)
largos = {p: np.array([len(i) for i in tok(t.texto.tolist())['input_ids']]) for p, t in partes.items()}
filas = []
for L in [128, 256, 512]:
    neg = partes['train'].sentimiento.eq('negativa').to_numpy()
    filas.append({'max_length': L, '% train truncado': (largos['train'] > L).mean() * 100,
                  '% negativas de train truncadas': (largos['train'][neg] > L).mean() * 100, '% externo truncado': (largos['externo'] > L).mean() * 100})
print('Subpalabras por reseña en train (percentiles 50/90/95/99):', np.percentile(largos['train'], [50, 90, 95, 99]).round(0))
display(pd.DataFrame(filas).set_index('max_length').round(1))

In [ ]:
# 2. Fine-tuning: pérdida ponderada por clase (igual que el baseline balanceado), lr 2e-5, batch 16, 3 épocas,
#    weight decay 0,01, warmup 10 %, se conserva la época con mejor F1 macro de validación
pesos = torch.tensor(compute_class_weight('balanced', classes=np.arange(3), y=y['train']), dtype=torch.float)
CALENTAMIENTO = ({'warmup_ratio': 0.1} if 'warmup_ratio' in inspect.signature(TrainingArguments.__init__).parameters
                 else {'warmup_steps': 0.1})          # transformers 5 expresa la proporción como warmup_steps < 1

class TrainerPonderado(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=pesos.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def a_dataset(parte, L):
    d = Dataset.from_dict({'texto': partes[parte].texto.tolist(), 'label': y[parte].tolist()})
    return d.map(lambda b: tok(b['texto'], truncation=True, max_length=L), batched=True, remove_columns=['texto'])

def f1_eval(ev):
    logits, labels = ev
    return {'f1_macro': f1_score(labels, logits.argmax(-1), average='macro')}

def probabilidades(modelo, textos, L, lote=64):
    modelo.eval(); dev = next(modelo.parameters()).device; salida = []
    with torch.inference_mode():
        for i in range(0, len(textos), lote):
            enc = tok(list(textos[i:i + lote]), truncation=True, max_length=L, padding=True, return_tensors='pt').to(dev)
            salida.append(torch.softmax(modelo(**enc).logits.float(), -1).cpu().numpy())
    return np.vstack(salida)

def entrenar(L, semilla):
    set_seed(semilla)
    modelo = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=3, id2label=id2label, label2id=label2id)
    args = TrainingArguments(output_dir=f'tmp_beto_{L}_{semilla}', learning_rate=2e-5, per_device_train_batch_size=16,
                             per_device_eval_batch_size=64, num_train_epochs=3, weight_decay=0.01, **CALENTAMIENTO,
                             eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, save_only_model=True,
                             load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True,
                             logging_strategy='epoch', seed=semilla, report_to='none', disable_tqdm=True,
                             fp16=torch.cuda.is_available())
    kw = dict(model=modelo, args=args, train_dataset=a_dataset('train', L), eval_dataset=a_dataset('val', L),
              data_collator=DataCollatorWithPadding(tok), compute_metrics=f1_eval)
    try:
        trainer = TrainerPonderado(processing_class=tok, **kw)
    except TypeError:                                   # versiones anteriores de transformers
        trainer = TrainerPonderado(tokenizer=tok, **kw)
    trainer.remove_callback(PrinterCallback)
    t0 = time.perf_counter(); trainer.train(); seg = time.perf_counter() - t0
    evals = [h for h in trainer.state.log_history if 'eval_f1_macro' in h]
    modelo = trainer.model
    shutil.rmtree(args.output_dir, ignore_errors=True); del trainer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    pv = probabilidades(modelo, partes['val'].texto.tolist(), L)
    return modelo, pv, {'max_length': L, 'semilla': semilla, 'f1_val': f1_score(y['val'], pv.argmax(1), average='macro'),
                        'recall_negativa_val': recall_score(y['val'], pv.argmax(1), labels=[0], average=None)[0],
                        'f1_val_por_epoca': [round(h['eval_f1_macro'], 4) for h in evals], 'segundos': seg}

CORRIDAS, MODELOS = [], {}
for L in LONGITUDES:
    print(f'Entrenando BETO · max_length {L} · semilla {SEED} …', flush=True)
    modelo, pv, info = entrenar(L, SEED)
    CORRIDAS.append(info); MODELOS[L] = (modelo.to('cpu'), pv)
    print(f"   F1 val {info['f1_val']:.3f} · por época {info['f1_val_por_epoca']} · {info['segundos'] / 60:.1f} min", flush=True)
display(pd.DataFrame(CORRIDAS).round(3))

In [ ]:
# 3. Selección con VALIDACIÓN (empate a 3 decimales → la longitud menor, más barata) y repetición con otras semillas
L_ELEGIDA = sorted(CORRIDAS, key=lambda c: (-round(c['f1_val'], 3), c['max_length']))[0]['max_length']
modelo, pv = MODELOS[L_ELEGIDA]
for L in LONGITUDES:
    if L != L_ELEGIDA: del MODELOS[L]
gc.collect()
print('Longitud elegida en validación:', L_ELEGIDA)
EXTRA = []
for s in SEMILLAS_EXTRA:
    print(f'Repitiendo max_length {L_ELEGIDA} con la semilla {s} …', flush=True)
    m_s, pv_s, info = entrenar(L_ELEGIDA, s)
    m_s = m_s.to(DISPOSITIVO)
    info['f1_test'] = f1_score(y['test'], probabilidades(m_s, partes['test'].texto.tolist(), L_ELEGIDA).argmax(1), average='macro')
    EXTRA.append(info); del m_s; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"   F1 val {info['f1_val']:.3f} · F1 test {info['f1_test']:.3f}")

In [ ]:
# 4. Evaluación única en test y en la prueba externa (modelo de la semilla 42 con la longitud elegida)
modelo = modelo.to(DISPOSITIVO)
P = {p: probabilidades(modelo, partes[p].texto.tolist(), L_ELEGIDA) for p in ['val', 'test', 'externo']}
for p in ['test', 'externo']:
    print(f"{p}: F1 macro {f1_score(y[p], P[p].argmax(1), average='macro'):.3f} | recall negativa "
          f"{recall_score(y[p], P[p].argmax(1), labels=[0], average=None)[0]:.3f} | log loss {log_loss(y[p], P[p], labels=[0, 1, 2]):.3f}")
print(classification_report(y['test'], P['test'].argmax(1), target_names=CLASES, digits=3))
tabla = pd.concat([pd.DataFrame({'id': partes[p].id, 'parte': p, 'p_negativa': P[p][:, 0], 'p_neutra': P[p][:, 1], 'p_positiva': P[p][:, 2]})
                   for p in ['val', 'test', 'externo']], ignore_index=True)
tabla.to_csv(SALIDA / 'beto_probabilidades.csv', index=False)
print('Guardado:', SALIDA / 'beto_probabilidades.csv', tabla.shape)

In [ ]:
# 5. Robustez: los mismos contrafactuales y perturbaciones del notebook principal (mismo orden y misma semilla)
import sys; sys.path.insert(0, str(RAIZ))
from src.texto import normalizar, sin_tildes          # la misma limpieza del notebook principal y de la demo
casos = pd.DataFrame([
    ('afirmación', 'Sede recomendable. Las máquinas estaban limpias, el personal fue amable y las clases muy buenas. Volveré.', 'positiva'),
    ('negación', 'Sede nada recomendable. Las máquinas no estaban limpias, el personal no fue amable y las clases no eran buenas. No volveré.', 'negativa'),
    ('orden A (mismas palabras)', 'No es malo, es bueno', 'positiva'),
    ('orden B (mismas palabras)', 'No es bueno, es malo', 'negativa'),
    ('contraste', 'Buena ubicación. La ubicación es buena, pero las duchas estaban sucias y había mucho ruido. No la recomiendo.', 'negativa'),
    ('errores de escritura', 'Sede nada recomendavle. Las duchas no estavan linpias y el personal no fue amavle.', 'negativa'),
    ('mayúsculas y signos', 'PÉSIMO SERVICIO!!! NUNCA HAY AGUA CALIENTE Y NADIE RESPONDE. NO VUELVO!!!', 'negativa'),
    ('ironía', 'Genial, solo tardaron dos horas en atenderme. Maravilloso.', 'negativa'),
    ('vocabulario de gimnasio', 'Las máquinas de cardio siempre están dañadas y nunca hay cupo en spinning. Pienso cancelar el plan.', 'negativa'),
    ('emoji', 'Todo perfecto 😍😍', 'positiva'),
    ('una palabra repetida', 'bien ' * 300, 'fuera de alcance'),
    ('otro idioma', 'Terrible gym. The showers were dirty and the staff was rude. We will not come back.', 'fuera de alcance'),
    ('fuera de dominio', 'La fotosíntesis depende de la luz', 'fuera de alcance'),
], columns=['caso', 'texto', 'esperado'])
pc = probabilidades(modelo, [normalizar(t) for t in casos.texto], L_ELEGIDA)
casos['beto'] = [f'{CLASES[i]} ({p:.2f})' for i, p in zip(pc.argmax(1), pc.max(1))]
casos.to_csv(SALIDA / 'beto_contrafactuales.csv', index=False)
display(casos.assign(texto=casos.texto.str.slice(0, 80)))

def con_errores(s, prob, rng):
    salida = []
    for w in s.split():
        if len(w) >= 5 and rng.random() < prob:
            i = int(rng.integers(1, len(w) - 2)); w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        salida.append(w)
    return ' '.join(salida)
test = partes['test']; rng = np.random.default_rng(SEED)
variantes = {'original': test.texto, 'todo en MAYÚSCULAS': test.texto.str.upper(), 'sin tildes': test.texto.map(sin_tildes),
             'errores de tecleo en 1 de cada 10 palabras': test.texto.map(lambda s: con_errores(s, .10, rng)),
             'errores de tecleo en 3 de cada 10 palabras': test.texto.map(lambda s: con_errores(s, .30, rng)),
             'sin título': test.resena, 'solo la primera mitad': test.texto.map(lambda s: ' '.join(s.split()[:max(len(s.split()) // 2, 1)]))}
ROBUSTEZ = {}
for nombre, textos in variantes.items():
    pred = probabilidades(modelo, textos.tolist(), L_ELEGIDA).argmax(1)
    ROBUSTEZ[nombre] = {'F1': f1_score(y['test'], pred, average='macro'), 'recall_negativa': recall_score(y['test'], pred, labels=[0], average=None)[0]}
display(pd.DataFrame(ROBUSTEZ).T.round(3))

In [ ]:
# 6. Costo: latencia por reseña en CPU (batch 1, con tokenización), rendimiento en lotes y tamaño del modelo
modelo_cpu = modelo.to('cpu').eval()
muestra = partes['test'].texto.tolist()[:200]
def una(t): return probabilidades(modelo_cpu, [t], L_ELEGIDA)
una(muestra[0]); tiempos = []
for t in muestra:
    t0 = time.perf_counter(); una(t); tiempos.append((time.perf_counter() - t0) * 1000)
t0 = time.perf_counter(); probabilidades(modelo_cpu, muestra[:160], L_ELEGIDA, lote=32); rps = 160 / (time.perf_counter() - t0)
dir_modelo = RAIZ / 'modelos' / 'beto'; modelo_cpu.save_pretrained(dir_modelo); tok.save_pretrained(dir_modelo)
tam = sum(f.stat().st_size for f in dir_modelo.rglob('*') if f.is_file()) / 1e6
corrida = next(c for c in CORRIDAS if c['max_length'] == L_ELEGIDA)
resumen = {'checkpoint': CHECKPOINT, 'max_length': L_ELEGIDA, 'semilla': SEED, 'corridas_validacion': CORRIDAS, 'semillas_extra': EXTRA,
           'f1_test_semillas': [f1_score(y['test'], P['test'].argmax(1), average='macro')] + [e['f1_test'] for e in EXTRA],
           'parametros': int(sum(p.numel() for p in modelo_cpu.parameters())), 'tamano_mb': round(tam, 1),
           'segundos_entrenamiento': corrida['segundos'], 'latencia_p50_ms_cpu': float(np.percentile(tiempos, 50)),
           'latencia_p95_ms_cpu': float(np.percentile(tiempos, 95)), 'resenas_por_segundo_cpu': rps, 'robustez': ROBUSTEZ,
           'gpu': GPU, 'nucleos_cpu': os.cpu_count(), 'torch': torch.__version__, 'transformers': transformers.__version__,
           'fecha': time.strftime('%Y-%m-%d %H:%M')}
(SALIDA / 'beto_resumen.json').write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=float), encoding='utf-8')
print(f"Latencia CPU p50 {resumen['latencia_p50_ms_cpu']:.0f} ms · p95 {resumen['latencia_p95_ms_cpu']:.0f} ms · {rps:.0f} reseñas/s en lotes de 32 · "
      f"{resumen['parametros'] / 1e6:.0f} M parámetros · {tam:.0f} MB en disco")
print('F1 test por semilla:', [round(v, 3) for v in resumen['f1_test_semillas']])

In [ ]:
# 7. Descargar los resultados para subirlos al repositorio (carpeta resultados/beto/)
if 'google.colab' in sys.modules:
    from google.colab import files
    for nombre in ['beto_resumen.json', 'beto_probabilidades.csv', 'beto_contrafactuales.csv']:
        files.download(str(SALIDA / nombre))
else:
    print('Archivos en', SALIDA)

**Después de ejecutar:** suba `beto_resumen.json`, `beto_probabilidades.csv` y `beto_contrafactuales.csv` a `resultados/beto/` del repositorio y vuelva a ejecutar el notebook 01. Los pesos ajustados (unos 440 MB en `modelos/beto/`) **no** se suben a GitHub: superan el límite de tamaño y no hacen falta para reproducir las cifras.